# 03. Przeglad i analiza podejsc

**Etap z planu pracy:** przeglad i analiza podejsc.

Ten notebook porownuje podejscia klasyczne, statystyczne, ML i transformerowe dla trzech hipotez.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "database").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205_prepared.csv"
RAW_DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
METADATA_PATH = PROJECT_ROOT / "outputs" / "prepared_dataset_metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "czysta_baza"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("Prepared data exists:", DATA_PATH.exists())
print("Output dir:", OUTPUT_DIR)

assert DATA_PATH.exists(), f"Brakuje pliku z przygotowana baza: {DATA_PATH}"


PROJECT_ROOT: C:\Users\szymon\projekt_reddit
DATA_PATH: C:\Users\szymon\projekt_reddit\database\NajnowszaWersjaBazy1205_prepared.csv
Prepared data exists: True
Output dir: C:\Users\szymon\projekt_reddit\outputs\czysta_baza


In [2]:
methods = pd.DataFrame([
    {
        "hypothesis": "H1 eskalacja emocjonalna",
        "approach": "Tabela kontyngencji + test Fishera",
        "features": "high_previous_anger_24h, is_negative_link",
        "why": "Najprostszy test roznicy udzialow dla rzadkiego zdarzenia.",
        "main_metric": "odds ratio, p-value, roznica odsetkow",
    },
    {
        "hypothesis": "H1 eskalacja emocjonalna",
        "approach": "Regresja logistyczna jako rozszerzenie",
        "features": "prev_pair_mean_anger_24h, prev_pair_interactions_24h, cechy kontrolne",
        "why": "Pozwala kontrolowac historie pary i inne predyktory.",
        "main_metric": "wspolczynnik anger, ROC/F1 pomocniczo",
    },
    {
        "hypothesis": "H2 zlozonosc poznawcza",
        "approach": "Porownanie grup + Mann-Whitney U",
        "features": "Average word length, LIWC_Conj, ARI, LIWC_CogMech, Number of words",
        "why": "Cechy sa ciagle i moga miec rozklady nienormalne.",
        "main_metric": "roznica srednich, Cohen d, p-value",
    },
    {
        "hypothesis": "H3 model multimodalny",
        "approach": "TF-IDF + Logistic Regression balanced",
        "features": "combined_text",
        "why": "Mocny klasyczny baseline tekstowy dla krotkich i srednich tekstow.",
        "main_metric": "negative_f1, macro_f1",
    },
    {
        "hypothesis": "H3 model multimodalny",
        "approach": "Cechy numeryczne/LIWC/sieciowe + Random Forest",
        "features": "numeric_features, structural_features",
        "why": "Sprawdza nieliniowe zaleznosci bez reprezentacji tekstowej TF-IDF.",
        "main_metric": "negative_f1, recall klasy negatywnej",
    },
    {
        "hypothesis": "H3 model multimodalny",
        "approach": "Hugging Face baseline i opcjonalne embeddingi",
        "features": "Content_Sentiment, Content_Score, opcjonalne sentence-transformers",
        "why": "Pozwala porownac gotowy sygnal transformerowy z cechami recznie wyliczonymi.",
        "main_metric": "negative_f1, macro_f1",
    },
])
display(methods)
methods.to_csv(OUTPUT_DIR / "03_methods_review.csv", index=False)


,hypothesis,approach,features,why,main_metric
0,H1 eskalacja emocjonalna,Tabela kontyngencji + test Fishera,"high_previous_anger_24h, is_negative_link",Najprostszy test roznicy udzialow dla rzadkieg...,"odds ratio, p-value, roznica odsetkow"
1,H1 eskalacja emocjonalna,Regresja logistyczna jako rozszerzenie,"prev_pair_mean_anger_24h, prev_pair_interactio...",Pozwala kontrolowac historie pary i inne predy...,"wspolczynnik anger, ROC/F1 pomocniczo"
2,H2 zlozonosc poznawcza,Porownanie grup + Mann-Whitney U,"Average word length, LIWC_Conj, ARI, LIWC_CogM...",Cechy sa ciagle i moga miec rozklady nienormalne.,"roznica srednich, Cohen d, p-value"
3,H3 model multimodalny,TF-IDF + Logistic Regression balanced,combined_text,Mocny klasyczny baseline tekstowy dla krotkich...,"negative_f1, macro_f1"
4,H3 model multimodalny,Cechy numeryczne/LIWC/sieciowe + Random Forest,"numeric_features, structural_features",Sprawdza nieliniowe zaleznosci bez reprezentac...,"negative_f1, recall klasy negatywnej"
5,H3 model multimodalny,Hugging Face baseline i opcjonalne embeddingi,"Content_Sentiment, Content_Score, opcjonalne s...",Pozwala porownac gotowy sygnal transformerowy ...,"negative_f1, macro_f1"


## Wybor metod do implementacji

W kolejnym notebooku uruchamiamy wersje rdzeniowe:

- H1: test Fishera, chi-kwadrat i roznica odsetkow.
- H2: porownanie grup, Cohen d, korelacja punktowo-dwuseryjna i Mann-Whitney U.
- H3: benchmark kilku modeli na stalym podziale chronologicznym.

Taki zestaw daje jednoczesnie interpretowalnosc hipotez H1-H2 i praktyczny test predykcyjny H3.


In [3]:
RUN_OPTIONAL_TRANSFORMERS_DEMO = False
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

if RUN_OPTIONAL_TRANSFORMERS_DEMO:
    from transformers import pipeline

    df = pd.read_csv(DATA_PATH, nrows=20)
    texts = (df["Raw_Title"].fillna("").astype(str) + " " + df["Raw_Content"].fillna("").astype(str)).tolist()
    sentiment_pipe = pipeline("sentiment-analysis", model=MODEL_NAME, truncation=True)
    display(pd.DataFrame(sentiment_pipe(texts[:5])))
else:
    print(
        "Opcjonalny pokaz nowego modelu Hugging Face jest wylaczony. "
        "Ustaw RUN_OPTIONAL_TRANSFORMERS_DEMO = True, jesli srodowisko ma dostep do modelu."
    )


Opcjonalny pokaz nowego modelu Hugging Face jest wylaczony. Ustaw RUN_OPTIONAL_TRANSFORMERS_DEMO = True, jesli srodowisko ma dostep do modelu.


## Rezultat etapu

Do finalnego porownania wybieramy metody, ktore mozna odtworzyc lokalnie bez pobierania duzych modeli. Komorki transformerowe zostaja jako opcjonalne rozszerzenie badania.
